# Incorrect Mental Models About High Availability and Network Routing

While designing a Spotify-like system for hundreds of millions of users, I realized that several incorrect mental models had prevented me from understanding how large-scale systems avoid Single Points of Failure (SPOFs).

The biggest breakthrough was realizing that **a logical address does not necessarily correspond to a single physical machine or a single physical path.**

### 1. Mistake: One IP Address = One Physical Machine

**An IP address is a logical network address, not the identity of a specific physical machine.**

A destination IP can be reachable through multiple network paths.

Routers can use routing tables and mechanisms such as ECMP (Equal-Cost Multi-Path) to distribute connections across multiple next hops.

```text
              One Public IP
                    |
            Multiple Routes
               /        \
          Router A     Router B
            /   \        /   \
           LB1  LB2     LB3  LB4
```

A single logical service address can be served by multiple physical machines.

---

## 2. Mistake: One Logical Entry Point = One Physical Entry Point

### My old mental model

If millions of users accessed `spotify.com`, I assumed all their requests must eventually pass through one physical entry point.

```text
Millions of Users
        |
   spotify.com
        |
  One Entry Machine  <-- SPOF!
        |
    LB Cluster
```

I confused a **single logical identity** with a **single physical dependency**.

### Correct mental model

A stable logical entry point can map to multiple physical entry points.

For example:

```text
               spotify.com
                    |
                   DNS
                    |
          +---------+---------+
          |         |         |
         IP A      IP B      IP C
          |         |         |
       Entry A   Entry B   Entry C
```

DNS can resolve the same domain to different addresses based on routing policies, health, and geographic information.

---

## 3. Mistake: network is not a tree but a graph

### My old mental model

```text
          Governor
              |
       +------+------+
       |             |
      LB1           LB2
```

**What happens if the governor dies?**

I could add another governor:

```text
        Super Governor
              |
           Governor
              |
           LB Pool
```

But what happens if the super governor dies?

This creates an infinite regress.

### Correct mental model

Instead of building an increasingly tall hierarchy, we can build redundant paths and distributed routing mechanisms.

```text
             Internet
             /      \
         Router A  Router B
             \      /
             LB Pool
```

Neither router needs to be the permanent leader of the entire system.

Routing protocols and redundant network infrastructure allow traffic to use alternative paths when components fail.

But the Internet is a graph of interconnected networks, not one global tree with a single root router.

---

## 4. Mistake: More Machines = More Fault Tolerance

### My old mental model

I assumed:

```text
10 Routers > 1 Router

Therefore:

10 Routers = High Availability
```

I thought that as long as at least one of the ten routers remained alive, the system would continue functioning.

This ignores shared dependencies and remaining capacity.

### Correct mental model

**Fault tolerance depends on independent end-to-end paths, not merely the number of machines.**

Four examples make this clear.

**Example A: Ten routers connected to the same ISP**

```text
             Internet
                |
             ISP A        <-- SPOF
                |
     +----------+----------+
     |          |          |
  Router 1   Router 2   Router 10
```

If the ISP experiences a major outage, all ten routers may become unreachable even if every router is healthy.

**Example B: Ten routers sharing the same physical fiber route**

```text
             Internet
                |
        Shared Fiber     <-- SPOF
                |
       10 Healthy Routers
```

One construction accident that cuts the shared fiber route could disconnect every router simultaneously.

**Example C: Ten routers in the same data center**

```text
             Data Center
          Shared Power Supply
                  |
           10 Routers
```

If the data center loses power and backup power also fails, all ten routers could become unavailable.

**Example D: Nine routers fail, leaving one**

```text
Before:
10 Routers × 10 Gbps = 100 Gbps

After:
1 Router × 10 Gbps = 10 Gbps
```

The service may still be reachable, but the remaining router cannot necessarily handle all incoming traffic.

**Availability is not the same as sufficient capacity.**

### The real design principle

We need to consider:

- **Failure domains:** Can one event take down multiple supposedly redundant components?
- **Independent paths:** Are there genuinely alternative routes from users to the service?
- **Failover:** Can traffic move to healthy paths when a component fails?
- **Remaining capacity:** Can surviving components handle the redirected traffic?

The correct model is:

```text
                  Internet
                  /      \
               ISP A    ISP B
                 |        |
             Router A  Router B
                 \        /
           Redundant Network
                    |
                 LB Pool
```

Even this architecture is not absolutely failure-proof, but it reduces shared points of failure.

**Redundancy is meaningful only when the redundant components do not all depend on the same thing.**

---

## 5. Mistake: The Internet Automatically Handles Load Balancing

### My old mental model

Once I understood that one IP could represent multiple machines, I assumed I could simply assign the same IP to 100 load balancers and connect them to the Internet.

```text
100 LBs with the same IP
           |
        Internet

Automatic Load Balancing?
```

### Correct mental model

**An IP address identifies a destination, but it does not automatically define how traffic is distributed among machines.**

The network needs explicit routing mechanisms.

For example:

- BGP advertises network reachability.
- Routers maintain routing tables.
- ECMP distributes traffic across eligible next hops.
- Health monitoring and route withdrawal help remove failed paths.
- Redundant network infrastructure provides alternative routes.

In a self-managed data center, I must design or arrange these mechanisms.

With a managed cloud load balancer, much of this complexity is handled by the cloud provider.